<a id="unit-u.7acda996f9b942a0859f132310fe8318"></a>

<a id="investigate-the-payment-failures-in-grafana"></a>

# Grafana-да төлем ақауларын тексеру

**3-модуль: Жүйені бақылау деректерін визуализациялау** · [Сабақ](course.md) · [Тест](quiz03_observability_visualization.ipynb)

1-модульден бері дүкен тапсырыстарының шамамен жартысы сәтсіз болады. Бұл зертханалық жұмыста ақауды кезекші инженер сияқты Grafana-да тексересіз: журналдардан қате іздейсіз, дүкен қатынас журналынан сәтсіз checkout табасыз, біреуінің трассасын ашып, ақау басталған қатені анықтайсыз. Хост CPU-ы себеп емес екенін тексеріп, инцидентті бірден көрсететін бақылау тақтасын құрасыз. Grafana-дағы әр қадамнан кейін ноутбук ұяшығы соны SQL-мен тексереді; оны кез келген MySQL клиентінде де орындауға болады.

Шамамен 45 минут жоспарлаңыз. Төлем ақауы қосылған 1-модульдің DOG стегі мен демосы қажет (1-зертханалық жұмыстың 5-бөлімі). Көп қадам браузердегі Grafana-да орындалады; олардың айналасындағы ұяшықтарды ретімен орындаңыз. Қадамдар курс орнататын Grafana 11.4.0 және Doris App 1.0.5 плагиніне арналған.

<a id="unit-u.0bb09c12eb7f48e28296b0da53a0d51e"></a>

<a id="initialize-the-lab"></a>

### Зертханалық жұмысты инициализациялау

`lab.sql(...)` не басқа көмекшілерді қолданбас бұрын осы ұяшықты бір рет орындаңыз.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)

<a id="unit-u.e0e3347843004370978a1a1a2f3f6103"></a>

<a id="1-check-that-orders-are-failing"></a>

## 1. Тапсырыстардың сәтсіз болатынын тексеру

Тексеруге соңғы минуттардағы сәтсіз тапсырыстар қажет. Соңғы 15 минуттағы тапсырыстарды және қаншасы сәтсіз болғанын санаңыз:

In [ ]:
orders = lab.wait_for_rows(
    """
    SELECT COUNT(*) AS orders,
           SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS failed_orders,
           MAX(timestamp) AS latest_order
    FROM otel_traces
    WHERE service_name = 'checkout'
      AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
      AND timestamp >= UTC_TIMESTAMP() - INTERVAL 15 MINUTE
    HAVING SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) > 0
    """,
    title="Orders and failed orders of the last 15 minutes",
    hint="No order failed in the last 15 minutes. Check that the demo runs (lab.start_demo() in Module 1) and "
         "that paymentFailure is 50% (section 5 of Lab 1), wait a minute or two, then rerun this cell.",
    timeout_seconds=180,
)

<a id="unit-u.c28e715326c74ee1a05421d36affff3c"></a>

**Күтілетін нәтиже**

Бір жол: жүктеме генераторының соңғы 15 минуттағы тапсырыстары, олардың шамамен жартысы сәтсіз, және соңғы тапсырыстың UTC уақыты. Демо кемінде 15 минут жұмыс істесе, әдетте 30–60 тапсырыс болады. Демоны не жалаушаны 15 минуттан аз уақыт бұрын қоссаңыз, тапсырыс та, сәтсіздері де азырақ: мысалы 1 және 2-зертханалық жұмыстардан кейін бірден 7 тапсырыстың 4-еуі сәтсіз болуы мүмкін.

<a id="unit-u.be50850c4a7c4823a40dc149cd1d1c72"></a>

<a id="2-open-discover-and-set-the-time-range"></a>

## 2. Discover ашу және уақыт аралығын орнату

Doris App — Doris телеметриясын іздейтін Grafana плагині. Оның **Discover** беті бір кестенің жолдарын іздейді. [http://127.0.0.1:23000/a/velodb-doris-app/discover](http://127.0.0.1:23000/a/velodb-doris-app/discover) адресін (немесе `DORIS_OBS_GRAFANA_PORT` портын) ашыңыз. Іздеу үшін жүйеге кіру қажет емес: Grafana келушілерге беттерін көруге мүмкіндік береді.

Беттің жоғарғы жолағы нені іздейтінін анықтайды:

| Өріс | Міндеті | Бұл жұмыста |
| --- | --- | --- |
| **Datasource** | Doris-ке қосылатын Grafana дереккөзі | `Doris` |
| **Database**, **Table** | Ізделетін кесте | `otel`, `otel_logs` |
| **Mode** | Іздеу жазу тәсілі: қысқа `field:value` синтаксисі **Lucene** не `WHERE` шарты **SQL** | `Lucene` |
| Іздеу өрісі | Іздеудің өзі; өріс кеңесі режим синтаксисін көрсетеді | Әзірге бос |
| **Time Field** | Уақыт аралығы қолданылатын баған | `timestamp` |
| **Time Range** | Уақыт бойынша қай жолдарды іздеу | Келесі ұяшық шығаратын екі мән |
| **Query** | Іздеуді орындайды | |

Бет соңғы кірудегі баптауларды есте сақтайды, сондықтан әр ашқанда тексеріңіз.

Doris телеметрия уақытын UTC бойынша сақтайды, бет көрсететін жолдар да сондай. Бірақ Doris App уақыт аралығын компьютеріңіздің уақыт белдеуіндегідей салыстырады. Компьютеріңіз UTC қолданбаса, **Last 15 minutes** сияқты салыстырмалы аралық қате кезеңнен іздеп, жаңа жолдарды өткізіп алады не ештеңе таппайды. Оның орнына абсолют UTC аралығын орнатыңыз. Келесі ұяшық 30 минут бұрын басталып, қазірден 30 минут кейін аяқталатын аралық шығарады; жұмыс істеп жатқанда келген жолдар да қамтылады:

In [ ]:
from datetime import datetime, timedelta, timezone

now = datetime.now(timezone.utc).replace(tzinfo=None, microsecond=0)
TIME_FROM = str(now - timedelta(minutes=30))
TIME_TO = str(now + timedelta(minutes=30))
print("From:", TIME_FROM)
print("To:  ", TIME_TO)

<a id="unit-u.c34729cea42446daacd8a1668c29e7cf"></a>

**Күтілетін нәтиже**

UTC бойынша арасы бір сағат екі уақыт, мысалы `From: 2026-10-20 19:35:12` және `To: 2026-10-20 20:35:12`. Төмендегі SQL ұяшықтары сол аралықты қолданады, сондықтан сандары бетпен сәйкес.

Discover ішінде:

1. **Time Range** батырмасын басыңыз (алғаш кіргенде **Last 1 day** көрсетеді). **Absolute time range** ішінде **From** мәнін (`now-1d`) жоғарыдағы From уақытымен, **To** мәнін (`now`) To уақытымен алмастырып, **Apply time range** басыңыз.
2. **Table** — `otel_logs`, **Mode** — **Lucene**, іздеу өрісі бос екенін тексеріңіз.
3. **Query** басыңыз.

Бет сәйкес жолдарды (**hits**) санап, уақыт аралықтары бойынша сызады және ең жаңасын алдымен, әрқайсысын бағандарымен тізеді. Тізімдегі әр `trace_id` — сілтеме. To уақытынан асып жұмыс істесеңіз, алдыңғы ұяшықты қайта орындап, жаңа аралықты орнатыңыз.

<a id="unit-u.5e7bf417aa4a45acbf1c8e99f7d29c80"></a>

<a id="3-look-for-error-logs"></a>

## 3. Қате журналдарын іздеу

Тексерудің алғашқы іздеуі көбіне қателерге арналған. `severity_text` — әр дереккөз жазатын `ERROR` не `Error` сияқты белгі; `severity_number` барлық дереккөзді ортақ OpenTelemetry шкаласына келтіреді: онда 17–20 — ERROR, 21–24 — FATAL. Іздеу өрісіне мынаны теріңіз:

```text
severity_number:>=17
```

Содан кейін **Query** басыңыз. Ноутбук дәл сол іздеуді сервис бойынша санап SQL-мен орындайды:

In [ ]:
lab.sql(f"""
SELECT service_name, COUNT(*) AS error_logs, MAX(timestamp) AS latest_error
FROM otel_logs
WHERE timestamp >= '{TIME_FROM}' AND timestamp < '{TIME_TO}'
  AND severity_number >= 17
GROUP BY service_name
ORDER BY error_logs DESC
""", title="Error logs in the time range, by service");

<a id="unit-u.f53da02f5eb84bd4a8c8a9e9a303937f"></a>

**Күтілетін нәтиже**

Discover ең көбі бірнеше нәтиже, ұяшық сол санақтарды көрсетеді: 2-модуль жүктеген Doris FE қателері және демо басталғанда `accounting` жазған ықтимал бір `Consume error`. FE іске қосылу қателерін From уақытынан бұрын жазса, мүлде нәтиже жоқ. Ешқайсысы төлем туралы емес. Сәтсіз тапсырыстар қате журналы болып көрінбейді, өйткені payment пен checkout Collector-ға журнал жібермейді, ал дүкен проксиінің қатынас журналында маңыздылық деңгейі жоқ (2-модуль). Маңыздылық бойынша қате іздеу бұл инцидентті өткізіп алады, сондықтан қатынас журналынан іздеңіз.

<a id="unit-u.24a53e4e95d248a9963eee8b7dfff5f1"></a>

<a id="4-find-the-failed-checkouts-in-the-access-log"></a>

## 4. Қатынас журналынан сәтсіз checkout табу

Дүкен проксиі әр сұрауға әдіс, жол және HTTP күй коды бар бір қатынас журналы жолын жазады, мысалы `"POST /api/checkout HTTP/1.1" 500`. Сәтсіз тапсырыс — 500 күйімен аяқталған checkout сұрауы.

Lucene режимінде `body:` артындағы тырнақшалы мәтін `body` ішінде сөздері қатар әрі ретімен тұрған жолдарды табады. 500 күйімен аяқталған checkout сұрауларын іздеңіз:

```text
body:"POST /api/checkout" AND body:"HTTP/1.1 500"
```

`AND` екі фразаға да сәйкес жолдарды қалдырады. Жалғыз `"POST /api/checkout HTTP/1.1 500"` фразасы жүктеме генераторының тапсырыстарын табады, бірақ дүкен бетінде өзіңіз бергендерін өткізіп алады: бет жолға `/api/checkout?currencyCode=USD` сұрау параметрлерін қосады, оның сөздері `checkout` пен `HTTP` арасына түседі.

Кейін **Mode** мәнін **SQL** етіңіз. Іздеу өрісі енді SQL `WHERE` шартын қабылдайды. Жоғарыдағы Lucene іздеуі мына шартқа айналады:

```text
body MATCH_PHRASE 'POST /api/checkout' AND body MATCH_PHRASE 'HTTP/1.1 500'
```

Оны іздеу өрісіне енгізіп, **Query** қайта басыңыз: нәтижелер бірдей.

In [ ]:
lab.sql(f"""
SELECT COUNT(*) AS failed_checkouts, MIN(timestamp) AS first_failure, MAX(timestamp) AS latest_failure
FROM otel_logs
WHERE timestamp >= '{TIME_FROM}' AND timestamp < '{TIME_TO}'
  AND body MATCH_PHRASE 'POST /api/checkout'
  AND body MATCH_PHRASE 'HTTP/1.1 500'
""", title="Failed checkouts in the access log");

<a id="unit-u.0cc1cd0e2e3743e39a1d950259a00d62"></a>

**Күтілетін нәтиже**

Discover нәтижелерінің саны: жалаушаны қосқалы бері не жалауша одан бұрын қосылса From уақытынан бері әр сәтсіз тапсырысқа бір нәтиже, шамамен минутына біреу. `first_failure` жалауша сақталған уақытқа жақын, егер ол From уақытынан бұрын болмаса. Бұлар — 1-модульдің қате күйлі checkout span-дары санаған сол тапсырыстар: қатынас журналы мен трассалар бір ақауларды санайды.

`MATCH_PHRASE` әр жол мәтінін сканерлемейді. Doris экспорттаушысы `otel_logs` кестесін `body` кері индексімен құрған: ол әр мәтінді сөздерге бөліп, әр сөз бар жолдарды тіркейді. Doris фраза сөздері бар жолдарды тауып, сөздері қатар әрі фраза ретімен тұрғандарын қалдырады. Индекс дүкен беті жіберген жолдың басын қалай бөлетінін қараңыз:

In [ ]:
lab.sql("""
SELECT TOKENIZE('"POST /api/checkout?currencyCode=USD HTTP/1.1" 500 - via_upstream',
                '"parser" = "unicode", "lower_case" = "true"') AS tokens
""", title="The words that the index of otel_logs.body makes of an access log line");

<a id="unit-u.146f3107b71a4527b59ea27da53a050e"></a>

**Күтілетін нәтиже**

Сөздер тізімі: `post`, `api`, `checkout`, `currencycode`, `usd`, `http`, `1.1`, `500` және `via_upstream`. Индекс сөздерді кіші әріпке ауыстырып, тырнақша, қиғаш сызық, сұрақ белгісі мен дефисті алып тастайды; фразаға олар қажет емес және ол `POST` сөзін кез келген регистрде табады. `currencycode` пен `usd` — бір ұзын фразаның сәйкестігін бұзатын сөздер. 6-модуль индекс қанша оқуды үнемдейтінін өлшейді.

<a id="unit-u.d3a011735cf64b008a9a12759be2c5e2"></a>

<a id="5-open-the-trace-of-a-failed-checkout"></a>

## 5. Сәтсіз checkout трассасын ашу

Әр қатынас журналы жолында сұрауының трасса ID-і бар. Нәтижелер тізімінен кемінде бір минут бұрынғы жолдың `trace_id` мәнін басыңыз: сервистер span-дарды пакетпен экспорттайды, сондықтан ең жаңа сұрау span-дары Doris-ке әлі жетпеген болуы мүмкін. Оң жақта сұрау трассасын сарқырама түрінде көрсететін **Trace Panel** ашылады.

<img src="images/discover-failed-checkouts.png" alt="Doris App Discover беті: otel_logs кестесі, сәтсіз checkout үшін Lucene іздеуі, абсолют уақыт аралығы, аралық бойынша нәтижелер және HTTP 500 қатынас журналы жолдары мен трасса ID-лері">

Сарқыраманы оқу:

- Әр жол — span: сервис қалың қаріппен, операция және ұзақтық. Балалары астына шегініспен орналасады; оң жақ жолақ span-ның бүкіл сұрау ішінде қашан басталып, қанша уақыт алғанын көрсетеді.
- Қызыл белгі күйі қате span-ды көрсетеді. Панель уақытты компьютердің белдеуімен көрсетеді, сондықтан басталу уақыты Discover-дегі UTC уақытынан өзгеше болуы мүмкін.
- Жоғарыдағы **Span Filters** жолдарды шектейді; **Prev** пен **Next** сәйкес span-дарды кезекпен көрсетеді.

Бірінші жол — жүктеме генераторының өзі берген тапсырысы `POST`. Қате шақырылғаннан шақырушыға жоғары таралады, сондықтан прокси, frontend, checkout сияқты бірнеше сервисте белгі бар. Ең терең қате span-ына, **payment** `grpc.oteldemo.PaymentService/Charge` жолына жылжыңыз. Оның астындағы `charge` сияқты span-дарда қате жоқ. Жолды басыңыз. Астынан **Status** мәні `error` егжей-тегжейі ашылады. **Span Attributes** басып ашыңыз: `grpc` ішінде `"error_message":"Payment request failed. Invalid token. app.loyalty.level=gold"` бар. Ақау осы жерден басталады.

<img src="images/failed-checkout-trace.png" alt="Сәтсіз checkout Trace Panel көрінісі: checkout-тың payment шақыруында қызыл қате белгісі бар сарқырама және Status error мәнді payment Charge span-ының егжей-тегжейі; span атрибуттарындағы grpc ішінде Payment request failed қате хабары бар">

Скриншоттардың интерфейс белгілері: Search or jump to… — іздеу не өту; Sign in — кіру; Home — басты бет; Datasource — дереккөз; Database — дерекқор; Table — кесте; Mode — режим; Time Field — уақыт өрісі; Time Range — уақыт аралығы; Query — сұрауды орындау; Filter — сүзгі; Search — іздеу; Selected fields — таңдалған өрістер; Available fields — қолжетімді өрістер; hits — сәйкестіктер; Count — санақ; Auto — автоматты; Time Interval — уақыт интервалы; _source — бастапқы жазба. Trace Panel — трасса панелі; Give feedback — пікір жіберу; Trace ID — трасса ID-і; Export — экспорттау; Span Filters — span сүзгілері; spans — span саны; Prev/Next — алдыңғы/келесі; Service — сервис; Duration — ұзақтық; Start Time — басталу уақыты; Child Count — бала саны; Kind — түрі; Status — күй; error — қате; Library Name/Library Version — кітапхана атауы/нұсқасы; Span Attributes — span атрибуттары; Resource Attributes — ресурс атрибуттары; SpanID — span ID-і. Lucene/SQL, сервис/операция атаулары, өріс кілттері, уақыт белгілері, ID-лер және қате хабарының нақты мәтіні бастапқы техникалық мәндер күйінде қалады.

Ноутбук кемінде бір минут бұрынғы сәтсіз checkout қате span-дарын тізеді:

In [ ]:
failed = lab.query(f"""
SELECT trace_id
FROM otel_logs
WHERE timestamp >= '{TIME_FROM}' AND timestamp < '{TIME_TO}'
  AND body MATCH_PHRASE 'POST /api/checkout'
  AND body MATCH_PHRASE 'HTTP/1.1 500'
  AND timestamp < UTC_TIMESTAMP() - INTERVAL 1 MINUTE
ORDER BY timestamp DESC
LIMIT 1
""")
failed_trace = failed["trace_id"][0]
lab.sql(f"""
SELECT service_name, span_name, span_kind, status_message,
       events[1].name AS first_event,
       events[1].attributes['exception.message'] AS exception_message
FROM otel_traces
WHERE trace_id = '{failed_trace}'
  AND status_code = 'STATUS_CODE_ERROR'
ORDER BY service_name, span_name
""", title=f"Error spans of the trace {failed_trace}");

<a id="unit-u.3f7e14e6f7214b68a35ac75bedc120b4"></a>

**Күтілетін нәтиже**

Бес сервисте шамамен тоғыз қате span-ы: `checkout`, `frontend`, `frontend-proxy`, `load-generator` және `payment`. Сервистер қатені әртүрлі тіркейді. Payment сервисінің `grpc.oteldemo.PaymentService/Charge` span-ында `Payment request failed. Invalid token. app.loyalty.level=gold` хабарлы `exception` оқиғасы бар; панель оны span қатесінің хабары ретінде көрсетті. Checkout span-дары сол хабарды `status_message` ішінде `failed to charge card: could not charge the card:` мәтінінен кейін қайталайды; бір frontend span-ында сол мәтінді орайтын `exception` оқиғасы бар. Жол реті уақыт туралы ештеңе айтпайды: сұрау оларды сервис бойынша сұрыптайды. Қате қайдан басталғанын білу үшін сарқырамадағыдай ең терең қате span-ын табыңыз: балалары бар болса, олар қатесіз.

<a id="unit-u.e20398fc4c99457c96c3e3c2655d84f3"></a>

<a id="6-search-the-traces-page"></a>

## 6. Traces бетінен іздеу

Discover жолдарды, **Traces** беті толық трассаларды іздейді. Оны Grafana мәзіріндегі **Doris App** астындағы **Traces** арқылы не [http://127.0.0.1:23000/a/velodb-doris-app/traces](http://127.0.0.1:23000/a/velodb-doris-app/traces) адресінен ашыңыз. Ол `otel_traces` оқиды.

1. **Time Range** мәнін Discover-дегідей абсолют аралық етіп, **Apply time range** басыңыз.
2. **Service** пен **Operation** мәндерін **ALL** күйінде қалдырыңыз. Олар алғашқы span бойынша таңдайды; жүктеме генераторы тапсырыстарында бұл checkout span-ы емес, оның `POST` span-ы. Браузерде берілген тапсырыстың түбірі `frontend-web` болуы мүмкін.
3. **Tags** ішіне `http.status_code=500` теріп, **Enter** арқылы қолданыңыз. Тегтер — span атрибуттары: бұл `http.status_code` атрибуты `500` болатын span-ы бар трассаларды, мысалы сәтсіз checkout прокси span-ын табады.
4. **Find Traces** басыңыз.

Диаграмма әр трассаны басталу уақыты мен ұзақтығына қарай нүктемен көрсетеді. Тізім түбір сервис пен операцияны, трасса ID-ін, span санын, сервистерін және ұзақтығын көрсетеді. HTTP 500 сәтсіз тапсырыстан кең ұғым: демо `frontend` пен `payment` іске қосылу GET сұрауларын да тіркейді, олар 500 қайтарады және checkout span-ы жоқ. Оларда жиі бір не бірнеше span ғана болады. Инцидент үшін `checkout` пен `payment` екеуі де бар трассаны, мысалы `load-generator:POST` тапсырысын не браузерде берген тапсырысыңызды ашыңыз. 4-бөлімдегі сәтсіз checkout трассасының ID-і де оны тануға көмектеседі.

In [ ]:
lab.sql(f"""
WITH http_500 AS (
    SELECT DISTINCT trace_id
    FROM otel_traces
    WHERE timestamp >= '{TIME_FROM}' AND timestamp < '{TIME_TO}'
      AND span_attributes['http.status_code'] = '500'
), failed_orders AS (
    SELECT DISTINCT trace_id
    FROM otel_traces
    WHERE timestamp >= '{TIME_FROM}' AND timestamp < '{TIME_TO}'
      AND service_name = 'checkout'
      AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
      AND status_code = 'STATUS_CODE_ERROR'
)
SELECT COUNT(*) AS traces_with_http_500,
       COUNT(f.trace_id) AS failed_checkout_traces,
       COUNT(*) - COUNT(f.trace_id) AS other_http_500_traces
FROM http_500 AS h
LEFT JOIN failed_orders AS f ON h.trace_id = f.trace_id
""", title="HTTP 500 traces, split into failed checkouts and other requests");

<a id="unit-u.95131ff9437d498b98370b9391386883"></a>

**Күтілетін нәтиже**

Бір жол. `traces_with_http_500` беттегі жалпы іздеуге сәйкес. Ол `failed_checkout_traces` пен `other_http_500_traces` қосындысы; соңғысы нөлден өзге болуы мүмкін, өйткені іске қосылу сұраулары да 500 қайтарады. Тек сәтсіз checkout ішкі жиыны төлем инцидентіне жатады және 4-бөлімдегі сәтсіз checkout қатынас журналдарымен салыстырылуы тиіс. Түбір сервис пен операция артық сұрауларды тануға көмектеседі; жалғыз HTTP 500 себебін анықтамайды.

Іздеулер арасында келген жаңа трассалар аз айырма тудыруы мүмкін, бірақ checkout span-ы жоқ іске қосылу GET-і қанша күтсеңіз де басқа сұрау түрі болып қалады. Traces беті толық сұраулар мен ұзақтығы туралы, Discover жолдар туралы сұрақтарға жауап береді.

<a id="unit-u.529dd4e190354edbb4af745ad445de0f"></a>

<a id="7-rule-out-the-host"></a>

## 7. Хостты себептен алып тастау

Ақаулар шамадан тыс жүктелген машинадан болуы мүмкін бе? DOG Collector Docker хостының CPU-ын әр 15 секунд сайын өлшейді: macOS пен Windows-та — Docker Desktop Linux виртуалды машинасы, Linux-та — компьютеріңіздің өзі. Өлшемдерді `otel_metrics_gauge` кестесіне `system.cpu.utilization` метрикасы ретінде жазады.

Алдымен жолдарды қараңыз. Discover ішінде **Table** мәнін `otel_metrics_gauge` етіп, сол уақыт аралығын қалдырып, Lucene режимінде мынаны іздеңіз:

```text
metric_name:"system.cpu.utilization"
```

Жолды ашу үшін сол жағындағы көрсеткіні басыңыз; `attributes` ішінде `state = idle` барының бірін таңдаңыз. Doris App 1.0.5 ішінде ашылған сандық өріс мәні нөл болса `-` көрсетуі мүмкін. Оны жоқ дерек деп түсінбес бұрын жолдың бастапқы өрістерін не төмендегі SQL-ді тексеріңіз. `attributes` `{"cpu":"cpu3","state":"idle"}` сияқты бір CPU ядросы мен бір күйді атайды; `value` — сол ядроның алдыңғы өлшемнен бері осы күйде болған уақыт үлесі, 0–1 арасында. Сондықтан жұмысбасты хостта `idle` мәндері аз болады, ал бір жол бүкіл хост туралы ештеңе айтпайды.

Хосттың жұмысбасты уақыт үлесі — 1-ден барлық ядроның орташа `idle` үлесін шегеру. Бірнеше бағдарлама осы атаулы метриканы береді; DOG Collector жолдарында `resource_attributes` ішінде хост атауы бар (8-модуль басқаларын түсіндіреді). Минуттық жұмысбасты үлесті есептеңіз:

In [ ]:
lab.sql("""
SELECT DATE_FORMAT(timestamp, '%H:%i') AS minute_utc,
       ROUND((1 - AVG(value)) * 100, 1) AS host_cpu_busy_percent
FROM otel_metrics_gauge
WHERE metric_name = 'system.cpu.utilization'
  AND attributes['state'] = 'idle'
  AND scope_name LIKE '%hostmetricsreceiver%'
  AND resource_attributes['host.name'] IS NOT NULL
  AND timestamp >= UTC_TIMESTAMP() - INTERVAL 15 MINUTE
GROUP BY minute_utc
ORDER BY minute_utc
""", title="Host CPU busy, per minute, over the last 15 minutes");

<a id="unit-u.21f820e8c228405abd0d0b90c3d5495d"></a>

**Күтілетін нәтиже**

Жалаушаны қосқанға дейін де, кейін де әр минутқа бір жол, жұмысбастылық әдетте 5–15 пайыз. Хостта CPU қоры бар, ақаулар жүктемесін өзгертпеген: payment span-ы айтқандай, себеп қолданбада. 8-модуль бұл үлес 90 пайыздан жоғары болып тұрса ескертеді.

<a id="unit-u.d42a1b6ffdcb4a2ba4c1b86ec639a2b8"></a>

<a id="8-build-an-incident-dashboard"></a>

## 8. Инцидент бақылау тақтасын құру

Бақылау тақтасы тексеру сұрауларын өздігінен жаңаратын панельдерде сақтайды. Панельдері Grafana-ның Doris дереккөзі арқылы SQL-мен сұрайды. Олар уақытты дұрыс түрлендіреді, сондықтан тақтада **Last 30 minutes** сияқты салыстырмалы аралық кез келген уақыт белдеуінде жұмыс істейді.

Тақта құруға тіркелгі керек:

1. Grafana-ның оң жоғарғы жағындағы **Sign in** басыңыз. `admin` пайдаланушысымен, `admin` құпиясөзімен кіріңіз. Grafana жаңа құпиясөз сұраса, **Skip** басыңыз.

Нәтижеге бөлінген минуттық тапсырыстардың бірінші панелін қосыңыз:

2. Мәзірден **Dashboards** не [http://127.0.0.1:23000/dashboards](http://127.0.0.1:23000/dashboards) ашыңыз. **New**, **New dashboard**, кейін **Add visualization** басыңыз.
3. **Select data source** ішінде **Doris** басыңыз. Панель редакторы ашылады: жоғарыда панель, астында сұрау редакторы, оң жақта **Time series** визуализациясы бар панель параметрлері.
4. Сұрау редакторын **Builder** режимінен **Code** режиміне ауыстырып, сұрауды мынамен алмастырыңыз:

   ```sql
   SELECT $__timeGroupAlias(timestamp, 1m),
          SUM(IF(status_code = 'STATUS_CODE_ERROR', 0, 1)) AS succeeded,
          SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS failed
   FROM otel_traces
   WHERE $__timeFilter(timestamp)
     AND service_name = 'checkout'
     AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
   GROUP BY 1
   ORDER BY 1
   ```

   `$__timeFilter(timestamp)` және `$__timeGroupAlias(timestamp, 1m)` — Grafana макростары: Grafana оларды тақтаның уақыт аралығы шартымен және әр минуттың `time` атты басталу уақытымен алмастырады.
5. **Format** мәнін **Table** орнына **Time series** етіп, **Run query** басыңыз. `succeeded` және `failed` деген екі сызық пайда болады.
6. Оң жақтағы **Title** мәнін `Orders per minute` етіңіз. **Graph styles** астында **Style** мәнін **Bars**, **Stack series** мәнін **Normal** етіңіз: әр минут бағаны барлық тапсырысты сәтті және сәтсізге бөліп көрсетеді.
7. **Back to dashboard** басыңыз.

Хост CPU-ының екінші панелін қосыңыз:

8. **Add**, кейін **Visualization** басыңыз. Редактор Doris дереккөзімен ашылады.
9. **Code** режиміне ауысып, сұрауды төмендегімен алмастырыңыз, **Format** мәнін **Time series** етіп, **Run query** басыңыз:

   ```sql
   SELECT $__timeGroupAlias(timestamp, 1m),
          (1 - AVG(value)) * 100 AS busy_percent
   FROM otel_metrics_gauge
   WHERE $__timeFilter(timestamp)
     AND metric_name = 'system.cpu.utilization'
     AND attributes['state'] = 'idle'
     AND scope_name LIKE '%hostmetricsreceiver%'
     AND resource_attributes['host.name'] IS NOT NULL
   GROUP BY 1
   ORDER BY 1
   ```

10. **Title** мәнін `Host CPU busy (%)` етіп, **Back to dashboard** басыңыз.

Тақтаны сақтаңыз:

11. Оң жоғарғы жақта тақтаның уақыт аралығын **Last 30 minutes** етіңіз.
12. **Save dashboard** басып, **Title** мәнін `Payment incident` етіңіз де, **Save** басыңыз.
13. Тақтаны қайта ашқанда 30 минуттық аралық сақталуы үшін **Edit**, **Settings**, кейін **JSON Model** басыңыз. Жоғарғы деңгейдегі `time` қасиеті `{"from": "now-30m", "to": "now"}` екенін тексеріңіз. Қажет болса тек осы қасиетті өзгертіңіз, `panels` пен модельдің қалғанын өзгеріссіз қалдырыңыз. **Save changes** басыңыз. Grafana-ның бұл нұсқасында уақыт таңдағышынан аралық таңдап, тек панельдерді сақтау әдепкі аралықты орнатпай қалдыруы мүмкін.
14. **Dashboards** ашып, тізімнен **Payment incident** қайта таңдаңыз. Алдыңғы бет URL-індегі аралыққа тәуелсіз **Last 30 minutes** мәнімен ашылуы керек.

<img src="images/payment-incident-dashboard.png" alt="Соңғы 30 минуттағы Payment incident бақылау тақтасы: Host CPU busy панелінде шамамен 8 пайыз сызығы, Orders per minute панелінде сәтті және сәтсіз тапсырыстардың жинақталған бағандары бар">

Скриншот белгілері: Home — басты бет; Dashboards — бақылау тақталары; Payment incident — төлем инциденті; Search or jump to… — іздеу не өту; Share — бөлісу; Edit — өңдеу; Last 30 minutes — соңғы 30 минут; Refresh — жаңарту; Host CPU busy (%) — хост CPU жұмысбастылығы (%); Orders per minute — минуттық тапсырыстар; succeeded — сәтті; failed — сәтсіз. busy_percent — SQL нәтиже бағанының атауы. Тақта мен панель атауларын жоғарыдағы нақты ағылшынша күйінде енгізіңіз: ноутбук соларды тексереді.

Сақталған тақтаны Grafana-дан қайта оқыңыз:

In [ ]:
lab.grafana_dashboard("Payment incident");

<a id="unit-u.37188950ce824e75b7d005453c3e6aff"></a>

**Күтілетін нәтиже**

Сақталған әдепкі аралық — `from = now-30m`, `to = now`. Біреуі бос не басқа болса, жоғарыдағы 13–14-қадамдарды қайталаңыз.

Содан кейін әр панельге бірден екі жол: `Orders per minute` және `Host CPU busy (%)`; екеуі де `timeseries`, пішімі `time_series`, сонымен бірге енгізген SQL көрсетіледі. Кесте атауы тақтаға сілтейді. Ұяшық Grafana-да `Payment incident` атты тақта жоқ десе, тақтаны дәл осы атаумен сақтаңыз.

Тақтада жалаушаны қосқаннан кейінгі бағандардың көбінде `failed` бөлігі бар, ал хост CPU сызығы төмен қалады. Тақта Grafana томында сақталады: `lab.teardown(keep_data=True)` оны сақтайды, `lab.teardown()` жояды.

<a id="unit-u.36fd428ea2e846f687854d16976c6b3c"></a>

<a id="9-exercise-the-share-of-failed-charges"></a>

## 9. Жаттығу: сәтсіз төлемдер үлесі

Payment сервисі әр төлем сұрауын `grpc.oteldemo.PaymentService/Charge` атты сервер span-ы ретінде тіркейді. Соңғы 15 минутта payment қанша төлем сұрауын алғанын, қаншасы сәтсіз болғанын және сәтсіз үлесін пайызбен қайтаратын сұрау жазыңыз.

In [ ]:
# Your code here

<a id="unit-u.eb7fa2fece104983b91b935685e8a1a2"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған. Салыстыру үшін ашыңыз немесе орындаңыз.

In [ ]:
lab.sql("""
SELECT COUNT(*) AS charges,
       SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS failed_charges,
       ROUND(100 * SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) / COUNT(*), 1) AS failed_percent
FROM otel_traces
WHERE service_name = 'payment'
  AND span_name = 'grpc.oteldemo.PaymentService/Charge'
  AND timestamp >= UTC_TIMESTAMP() - INTERVAL 15 MINUTE
""", title="Charge requests of the last 15 minutes");

<a id="unit-u.63afabd28547491691a666d1a0893d09"></a>

**Күтілетін нәтиже**

Бір жол. Жалауша бүкіл 15 минут қосулы болса, `failed_percent` жалаушаның 50 мәніне жақын; бірнеше ондаған төлемде шамамен 35–65 аралығы қалыпты. Жалаушаны кейінірек қоссаңыз, үлес төменірек: бұрынғы төлемдердің бәрі сәтті болған.

<a id="unit-u.401cd06a0c764198b21ac8143a0b2e86"></a>

<a id="you-are-done-when"></a>

## Аяқтау шарттары

- Discover ішінде Lucene және SQL арқылы сәтсіз checkout сұрауларын таптыңыз, сандары ноутбукпен сәйкес;
- сәтсіз checkout трассасын ашып, payment сервисінің `Payment request failed. Invalid token. app.loyalty.level=gold` қатесін таптыңыз;
- хост CPU-ы 90 пайыздан едәуір төмен;
- `lab.grafana_dashboard("Payment incident")` екі панельді тізеді.

<a id="unit-u.33c27a5af5d0465caf037d50415b3572"></a>

<a id="next"></a>

## Әрі қарай

[3-тест: Жүйені бақылау деректерін визуализациялау](quiz03_observability_visualization.ipynb) файлын ашыңыз. Бұл — 1-деңгейдің соңғы зертханалық жұмысы; 2-деңгей [4-модуль: Жүйені бақылау деректерінің кесте модельдері](../../level2/module04-table-model/course.md) бөлімінен басталады.